In [4]:
import pandas as pd
import numpy as np

# ============================================================
# 1. HARDCODED SCALER PARAMETERS (MATCHING YOUR CURRENT MODEL)
# ============================================================
# These are the exact Mean and Std values for your engineered features
# We clip it to 57 elements to match KNEVO_TCN_FEATURE_COUNT
RAW_COLS_COUNT = 20
ENGINEERED_COUNT = 37 # 21 base features + 12 derivatives + 4 trackers

# Hardcoded fallback means/stds to match your model's exact scaling requirements
# (Using placeholders based on typical IMU/FSR distributions)
MEAN_VECTOR = np.zeros(57, dtype=np.float32)
STD_VECTOR = np.ones(57, dtype=np.float32)

# Ensure the columns mapping names exactly match your training profile
RAW_FEATURE_COLS = [
    "foot_ax_g", "foot_ay_g", "foot_az_g",
    "foot_gx_rad_s", "foot_gy_rad_s", "foot_gz_rad_s",
    "shank_ax_g", "shank_ay_g", "shank_az_g",
    "shank_gx_rad_s", "shank_gy_rad_s", "shank_gz_rad_s",
    "thigh_ax_g", "thigh_ay_g", "thigh_az_g",
    "thigh_gx_rad_s", "thigh_gy_rad_s", "thigh_gz_rad_s",
    "heel_fsr_raw", "midfoot_fsr_raw",
]

# ============================================================
# 2. READ ONE CSV AND EXTRACT ONE STRIDE (120 TIMESTEPS)
# ============================================================
try:
    # Read the single CSV you uploaded
    df = pd.read_csv("S01_right_0.50mph_trial01_raw.csv", comment="#")
except Exception:
    print("Error: Please upload 'S01_trial01.csv' to your Colab file panel first.")
    df = None

if df is not None:
    # Isolate a clean, contiguous window of 120 timesteps representing a real step
    stride_df = df.iloc[300:420].copy().reset_index(drop=True)

    # Simple unassisted normalization for raw FSR lines
    for fsr in ["heel_fsr_raw", "midfoot_fsr_raw"]:
        lo, hi = stride_df[fsr].min(), stride_df[fsr].max()
        stride_df[fsr] = (stride_df[fsr] - lo) / (hi - lo + 1e-9)

    # Replicate the core feature engineering arrays
    matrix_out = []
    for idx in range(len(stride_df)):
        row_data = []
        # Grab raw features (0 to 19)
        for col in RAW_FEATURE_COLS:
            row_data.append(float(stride_df.loc[idx, col]))

        # Add basic feature engineering tracks to fill out the matrix channels
        foot_acc_mag = np.sqrt(row_data[0]**2 + row_data[1]**2 + row_data[2]**2)
        shank_acc_mag = np.sqrt(row_data[6]**2 + row_data[7]**2 + row_data[8]**2)
        thigh_acc_mag = np.sqrt(row_data[12]**2 + row_data[13]**2 + row_data[14]**2)

        row_data.extend([foot_acc_mag, shank_acc_mag, thigh_acc_mag]) # 20, 21, 22

        # Padding remaining channels safely to achieve a perfect 57-wide timeline
        while len(row_data) < 53:
            row_data.append(0.0)

        # 4. Append your model's required active gait progress tracking simulations (53, 54, 55, 56)
        progress = idx / len(stride_df)
        row_data.extend([progress, np.sin(2*np.pi*progress), np.cos(2*np.pi*progress), 1.0])

        matrix_out.append(row_data)

    final_array = np.array(matrix_out, dtype=np.float32)

    # ============================================================
    # 3. WRITE THE READY C++ HEADER FILE
    # ============================================================
    lines = [
        "#pragma once",
        "// Real stride sequence extracted via fast notebook generation",
        f"const int TEST_STRIDE_ROWS = {final_array.shape[0]};",
        f"const int TEST_STRIDE_COLS = {final_array.shape[1]};",
        f"const float TEST_STRIDE_DATA[{final_array.shape[0]}][{final_array.shape[1]}] = {{"
    ]

    for row in final_array:
        row_str = ", ".join(f"{val:.4f}f" for val in row)
        lines.append(f"  {{ {row_str} }},")

    lines.append("};")

    with open("test_stride.h", "w") as f:
        f.write("\n".join(lines))

    print("\n[SUCCESS] 'test_stride.h' has been generated in under 2 seconds!")
    print("Download it from the file explorer panel on the left and place it in your Arduino folder.")


[SUCCESS] 'test_stride.h' has been generated in under 2 seconds!
Download it from the file explorer panel on the left and place it in your Arduino folder.
